# Classify Sample Delta Events

Train a PyTorch Lightning classifier from `data/sample_events`. The table contains one Delta snapshot with 100 financial transactions: 95 training rows and 5 test rows.


In [ ]:
from pathlib import Path
from typing import cast

import lightning as lightning
import torch
from torch import nn

from lit_deltalake.dataloaders import create_pytorch_dataloader
from lit_deltalake.readers.types import ColumnSpec, Filter

repository_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
table_uri = repository_root / "data" / "sample_events"
feature_columns = ("amount_usd", "deposit_amount", "current_balance", "is_test")
label_column = "event_type"
labels = ("deposit", "purchase", "transfer", "withdrawal")
train_filters = (Filter("is_test", "=", False),)
test_filters = (Filter("is_test", "=", True),)

In [ ]:
def to_float(value: object) -> float:
    return float(cast(float, value))


def event_type_to_index(value: object) -> int:
    return labels.index(cast(str, value))


class EventClassifier(lightning.LightningModule):
    def __init__(self, learning_rate: float = 1e-3) -> None:
        super().__init__()
        self.model = nn.Linear(len(feature_columns), len(labels))
        self.learning_rate = learning_rate
        self.loss_function = nn.CrossEntropyLoss()

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.model(features)

    def _shared_step(self, batch: dict[str, torch.Tensor], stage: str) -> torch.Tensor:
        features = torch.stack(tuple(batch[column].float() for column in feature_columns), dim=1)
        labels_batch = batch[label_column].long()
        logits = self(features)
        loss = self.loss_function(logits, labels_batch)
        accuracy = (logits.argmax(dim=1) == labels_batch).float().mean()
        self.log(f"{stage}_loss", loss, prog_bar=True)
        self.log(f"{stage}_accuracy", accuracy, prog_bar=True)
        return loss

    def training_step(self, batch: dict[str, torch.Tensor], batch_idx: int) -> torch.Tensor:
        return self._shared_step(batch, "train")

    def validation_step(self, batch: dict[str, torch.Tensor], batch_idx: int) -> None:
        self._shared_step(batch, "val")

    def test_step(self, batch: dict[str, torch.Tensor], batch_idx: int) -> None:
        self._shared_step(batch, "test")

    def configure_optimizers(self) -> torch.optim.Optimizer:
        return torch.optim.AdamW(self.parameters(), lr=self.learning_rate)

In [ ]:
columns = feature_columns + (label_column,)
column_specs = tuple(ColumnSpec(column, transform=to_float) for column in feature_columns) + (
    ColumnSpec(label_column, transform=event_type_to_index),
)
train_loader = create_pytorch_dataloader(
    str(table_uri),
    columns=columns,
    column_specs=column_specs,
    filters=train_filters,
    batch_size=32,
    shuffle_buffer_size=4096,
)
test_loader = create_pytorch_dataloader(
    str(table_uri),
    columns=columns,
    column_specs=column_specs,
    filters=test_filters,
    batch_size=32,
)
next(iter(train_loader))

In [ ]:
trainer = lightning.Trainer(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
)
model = EventClassifier()
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=test_loader)
trainer.test(model, dataloaders=test_loader)